In [9]:
import pandas as pd

In [10]:
filepath = "nrc_data_csv/german_nrc.csv"
df = pd.read_csv(filepath)
df.head(12)

,word,anger,disgust,fear,joy,sadness,surprise
0,verblüfft,0,0,0,0,0,0
1,Abakus,0,0,0,0,0,0
2,aufgeben,0,0,1,0,1,0
3,verlassen,1,0,1,0,1,0
4,Aufgabe,1,0,1,0,1,1
5,nachlassen,0,0,0,0,0,0
6,Minderung,0,0,0,0,0,0
7,Abb,0,0,0,0,0,0
8,Abt,0,0,0,0,0,0
9,abkürzen,0,0,0,0,0,0


In [11]:
from datasets import load_dataset
import pandas as pd

target_emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
random_state = 42

# Load German evaluation data

comb_df = pd.concat(
    map(pd.read_csv, ["xed_german.csv", "brighter_german.csv"]),
    ignore_index=True
)

comb_df["text"] = comb_df["text"].astype(str)
comb_df["emotion"] = comb_df["emotion"].astype(str)

print("Original German rows:", len(comb_df))
print("\nOriginal German class distribution:")
print(comb_df["emotion"].value_counts())


# Balance German data
target_n = comb_df["emotion"].value_counts().min()

balanced_parts = []
for emotion, group in comb_df.groupby("emotion"):
    balanced_parts.append(
        group.sample(n=target_n, random_state=random_state)
    )

comb_df = (
    pd.concat(balanced_parts, ignore_index=True)
    .sample(frac=1, random_state=random_state)
    .reset_index(drop=True)
)

print("\nBalanced German rows:", len(comb_df))
print("\nBalanced German class distribution:")
print(comb_df["emotion"].value_counts())

Original German rows: 6063

Original German class distribution:
emotion
joy         1729
anger       1218
sadness     1100
disgust      797
fear         640
surprise     579
Name: count, dtype: int64

Balanced German rows: 3474

Balanced German class distribution:
emotion
surprise    579
fear        579
disgust     579
sadness     579
anger       579
joy         579
Name: count, dtype: int64


In [12]:
import spacy
nlp = spacy.load("de_core_news_sm")

print(df)

             word  anger  disgust  fear  joy  sadness  surprise
0       verblüfft      0        0     0    0        0         0
1          Abakus      0        0     0    0        0         0
2        aufgeben      0        0     1    0        1         0
3       verlassen      1        0     1    0        1         0
4         Aufgabe      1        0     1    0        1         1
...           ...    ...      ...   ...  ...      ...       ...
14149        Zone      0        0     0    0        0         0
14150         Zoo      0        0     0    0        0         0
14151  zoologisch      0        0     0    0        0         0
14152    Zoologie      0        0     0    0        0         0
14153      Zoomen      0        0     0    0        0         0

[14154 rows x 7 columns]


In [13]:
import random

random.seed(42)

df["word"] = df["word"].astype(str).str.lower().str.strip()

df = (
    df.groupby("word", as_index=False)[target_emotions]
    .max()
)

lexicon = df.set_index("word")[target_emotions].to_dict(orient="index")

emotion_frequency = {}

for word, scores in lexicon.items():
    active_count = sum(scores[emotion] for emotion in target_emotions)

    if active_count > 0:
        emotion_frequency[word] = 1 / active_count
    else:
        emotion_frequency[word] = 0

predictions = []

for text in comb_df["text"]:

    count = {emotion: 0 for emotion in target_emotions}

    doc = nlp(text)

    for token in doc:

        candidates = [
            token.lemma_.lower().strip() if token.lemma_ else "",
            token.text.lower().strip()
        ]

        for candidate in candidates:

            if candidate in lexicon:

                word_weight = emotion_frequency[candidate]

                for emotion in target_emotions:
                    count[emotion] += (
                        lexicon[candidate][emotion] * word_weight
                    )

                break

    maxim = max(count.values())

    print(text)
    print(count)
    print()

    if maxim == 0:
        pred = "unknown"
    else:
        best_emotions = [emo for emo, score in count.items() if score == maxim]
        pred = random.choice(best_emotions)

    predictions.append(pred)

comb_df["prediction"] = predictions


Das ist nicht ungewohnlich.
{'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}

- Ich hoffe doch.
{'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}

„ich werde mit dieser afd, der angeblichen alternative, nicht koalieren und nicht mit ihr zusammenarbeiten. das ist ein versprechen das gebe ich ihnen hier heute ab. alles andere wäre unredlich!" #hartaberfair
{'anger': 0.0, 'disgust': 0.0, 'fear': 0.0, 'joy': 1.0, 'sadness': 0.0, 'surprise': 0.0}

also mir ging es im erste trimester damals doof und am ende wars halt anstrengend. hab den mutterschutz im studium nicht in anspruch genommen.  notfalls kannst du sicher abbrechen ohne probleme. klär das ab und geh sonst über die gleichstellungsbeauftragten oder so.
{'anger': 0.0, 'disgust': 0.0, 'fear': 0.5, 'joy': 1.0, 'sadness': 1.5, 'surprise': 0.0}

Wie ich sagte, schön für Sie, aber Sie werden es nicht mit mir tun.
{'anger': 0.0, 'disgust': 0.0, 'fear': 0.0, 'joy': 0.3333333333333333, 's

In [14]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

target_emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

def evaluate_lexicon_predictions(df, gold_col="emotion", pred_col="prediction"):
    """
    Evaluates lexicon predictions.
    Unknown predictions are counted as wrong in the main evaluation.
    Also reports coverage and covered-only performance.
    """

    eval_df = df.copy()
    eval_df[gold_col] = eval_df[gold_col].astype(str)
    eval_df[pred_col] = eval_df[pred_col].astype(str)

    # -------------------------
    # Main evaluation: unknown = wrong
    # -------------------------
    y_true = eval_df[gold_col]
    y_pred = eval_df[pred_col]

    print("=== Main evaluation: unknown counted as wrong ===")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Balanced accuracy:", round(balanced_accuracy_score(y_true, y_pred), 4))
    print("Macro F1:", round(f1_score(y_true, y_pred, labels=target_emotions, average="macro"), 4))
    print("Weighted F1:", round(f1_score(y_true, y_pred, labels=target_emotions, average="weighted"), 4))
    print()

    print(classification_report(
        y_true,
        y_pred,
        labels=target_emotions,
        digits=3,
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred, labels=target_emotions)
    cm_df = pd.DataFrame(cm, index=target_emotions, columns=target_emotions)

    print("Confusion matrix:")
    print(cm_df)
    print()

    # -------------------------
    # Coverage
    # -------------------------
    covered_mask = eval_df[pred_col] != "unknown"
    coverage = covered_mask.mean()

    print("=== Coverage ===")
    print("Coverage:", round(coverage, 4))
    print("Covered examples:", covered_mask.sum())
    print("Total examples:", len(eval_df))
    print()

    # -------------------------
    # Covered-only evaluation
    # -------------------------
    covered_df = eval_df[covered_mask].copy()

    if len(covered_df) > 0:
        print("=== Covered-only evaluation ===")
        print("Accuracy:", round(accuracy_score(covered_df[gold_col], covered_df[pred_col]), 4))
        print("Balanced accuracy:", round(balanced_accuracy_score(covered_df[gold_col], covered_df[pred_col]), 4))
        print("Macro F1:", round(f1_score(
            covered_df[gold_col],
            covered_df[pred_col],
            labels=target_emotions,
            average="macro",
            zero_division=0
        ), 4))
        print()

        print(classification_report(
            covered_df[gold_col],
            covered_df[pred_col],
            labels=target_emotions,
            digits=3,
            zero_division=0
        ))
    else:
        print("No covered examples.")

    results = {
        "accuracy_unknown_wrong": accuracy_score(y_true, y_pred),
        "balanced_accuracy_unknown_wrong": balanced_accuracy_score(y_true, y_pred),
        "macro_f1_unknown_wrong": f1_score(y_true, y_pred, labels=target_emotions, average="macro", zero_division=0),
        "weighted_f1_unknown_wrong": f1_score(y_true, y_pred, labels=target_emotions, average="weighted", zero_division=0),
        "coverage": coverage,
    }

    if len(covered_df) > 0:
        results.update({
            "accuracy_covered_only": accuracy_score(covered_df[gold_col], covered_df[pred_col]),
            "balanced_accuracy_covered_only": balanced_accuracy_score(covered_df[gold_col], covered_df[pred_col]),
            "macro_f1_covered_only": f1_score(
                covered_df[gold_col],
                covered_df[pred_col],
                labels=target_emotions,
                average="macro",
                zero_division=0
            ),
        })

    return results

In [15]:
results = evaluate_lexicon_predictions(
    comb_df,
    gold_col="emotion",
    pred_col="prediction"
)

=== Main evaluation: unknown counted as wrong ===
Accuracy: 0.188
Balanced accuracy: 0.188
Macro F1: 0.2128
Weighted F1: 0.2128

              precision    recall  f1-score   support

       anger      0.235     0.100     0.140       579
     disgust      0.265     0.100     0.145       579
        fear      0.265     0.164     0.203       579
         joy      0.362     0.352     0.357       579
     sadness      0.294     0.304     0.299       579
    surprise      0.173     0.107     0.132       579

   micro avg      0.279     0.188     0.224      3474
   macro avg      0.266     0.188     0.213      3474
weighted avg      0.266     0.188     0.213      3474

Confusion matrix:
          anger  disgust  fear  joy  sadness  surprise
anger        58       38    63   54       79        23
disgust      70       58    67   82      104        58
fear         36       20    95   75      108        57
joy          20       54    29  204       73        95
sadness      38       20    63   95

/Users/arsenimuravjov/miniforge3/envs/xed-llm/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/arsenimuravjov/miniforge3/envs/xed-llm/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
